# Notebook 2B: Choosing the Number of Clusters in K-means on METABRIC Data
*Authored by Dr. Noelle Anderson in 2026*

In Notebook 2A, you used K-means clustering to group similar patient samples and visualized those groups in PCA space. In this notebook, you will go one step further and ask an important question: **how many clusters should we use?**

You will use the same scaled METABRIC dataset from 2A and learn two common tools for choosing `k`, the number of clusters:
- **inertia**, which tells us how tightly data points sit around their cluster centers
- **silhouette score**, which tells us how well-separated the clusters are

In unsupervised learning, we usually do not start with known labels or one "correct answer." That means choosing `k` often involves comparing several kinds of evidence before choosing a defensible cluster number.

By the end of this notebook, you should be able to:
- compute and plot inertia across different values of `k`
- compute and compare silhouette scores
- explain why choosing `k` is often not completely straightforward
- compare clustering of different k values visually with PCA


## Notebook roadmap

In this notebook, you will:

1. Reload the scaled METABRIC dataset from Notebook 2A
2. Compute inertia for several values of `k`
3. Make and interpret an elbow plot
4. Compute silhouette scores for several values of `k`
5. Compare what the two methods suggest
6. Compare `k=3` and `k=5` in inertia, silhouette scores, and PCA space
7. Make a reasoned choice about `k`

# Step 0: Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score # new this time!

## Step 1: Load the scaled METABRIC data from Google Drive

We will use the same scaled METABRIC dataset from Notebook 2A. As before, we will mount Google Drive and load the file from the `SCIP2026ML` folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load the scaled METABRIC dataset from your Google Drive folder
file_path = "/content/drive/MyDrive/SCIP2026ML/metabric_1b_scaled.csv"

# patient_id should be used as the index, not as a feature
df = pd.read_csv(file_path, index_col="patient_id")

print("Shape of the scaled dataset:")
print(df.shape)

print("\nFirst 5 rows:")
display(df.head())

## How do we choose `k`?

In Notebook 2A, you saw that K-means needs us to choose the number of clusters, `k`, before we fit the model. We tried a couple of values and compared the results, but that raises an important question:

**How do we decide which value of `k` is reasonable?**

In real clustering work, there is usually not one perfect answer waiting to be discovered. Instead, we use a few tools to help us judge whether a clustering seems useful, clear, and reasonably structured.



---

One of the first tools we can use is called **inertia**.

**Inertia** tells us how tightly the data points are grouped around their assigned cluster centers.

To understand this, remember what K-means does:
- it creates `k` number of cluster centers, called **centroids**
- it assigns each patient to the nearest centroid
- it updates the centroids and repeats this process until the clusters settle down

Once that process is finished, inertia measures how far the points are from the centroid of the cluster they were assigned to. You can think of inertia as a measure of how much spread remains within the clusters after K-means assigns each point to a centroid.

So:
- **low inertia** means points tend to sit closer to their cluster centers
- this usually means the clusters are more **compact**, or more tightly grouped with less within-cluster spread

That sounds good, but there is an important limitation.

If we keep increasing `k`, inertia will usually go down. That happens because adding more clusters gives the algorithm more centroids, so each point has a better chance of being close to one of them. You can imagine an interia value of 0 which might seem really good, but it would just mean that each datapoint is now it's own centroid and cluster.

So lower inertia does not automatically mean we found the best or most meaningful clustering. Inertia is useful, but it cannot answer the whole question by itself, so we will explore multiple methods here.

## Step 2: Compute inertia for one clustering

In Notebook 2A, you already fit K-means models such as `k=2` and `k=3`. Now we will look at one of those models in a new way by asking:

**How compact are the clusters it made?**

We can start to answer that question with **inertia**.

Here, we will fit a K-means model with `k=3`, just like before, and then check its inertia value.

In [ ]:
# Fit a K-means model with k = 3
kmeans_3 = KMeans(n_clusters=3, random_state=42)
kmeans_3.fit(df)

# Print the inertia for this clustering
print("Inertia for k = 3:")

# Print the inertia value rounded to 3 decimal places using the f-string notation :.3f
print(f"{kmeans_3.inertia_:.3f}")

This gives us one inertia value for one clustering.

That number tells us how close the data points are to the centroids of the clusters they were assigned to. Lower inertia means the points are packed more tightly around their cluster centers.

But one inertia value by itself is hard to interpret, we do not yet know whether it is "low" or "high" for this dataset. To make inertia more useful, we need to compare it across several values of `k`.

## Step 3: Compare inertia across several values of `k`

Now we will check what happens to inertia when we change the number of clusters.

We will try values of `k` from 2 to 6. For each value, we will use a for loop to:
- fit a new K-means model
- record its inertia
- compare the results

This will help us see whether adding more clusters makes the data much more compact, or only a little more compact.

### <font color=0D9488>**Question 1:**</font> Below shows partial code for our for loop. It is your job to fill in the 5 missing `____` parts of the loop below.

In [ ]:
# Your solution here!
# Fill in the ____ blanks

# Values of k we want to compare
k_values = [2, 3, 4, 5, 6]

# Empty list to store the inertia values
inertia_values = []

for k in ____:
    # Create the K-means model
    kmeans = KMeans(n_clusters=____, random_state=42)

    # Fit the model to the data
    kmeans.____(df)

    # Add the inertia value to your list
    inertia_values.____(kmeans.____)

print("Inertia values:")
for k, inertia in zip(k_values, inertia_values): # zip is a handy way to pair up information in order
    print(f"k = {k}: inertia = {inertia:.3f}")

As you probably noticed, inertia goes down as `k` gets larger.

That makes sense because adding more clusters gives the algorithm more centroids, so each point has a better chance of being close to one of them.

But this also shows why inertia alone cannot fully answer the question of which `k` is best. If we only chased the lowest possible inertia, we would keep adding more and more clusters.

A common way to visualize this pattern is with an **elbow plot**.

## Step 4: Make an elbow plot

An **elbow plot** shows inertia on the y-axis and the number of clusters `k` on the x-axis.

We look for a place where the curve starts to bend or level off, that bend is often called the **elbow**.

The idea is that adding a few clusters at the beginning may greatly reduce inertia, because the algorithm can separate the data into more meaningful groups. After a certain point though, adding even more clusters may only reduce inertia by a small amount. If there is a visible elbow, it can suggest a reasonable place to stop adding clusters.

However, elbow plots do not always have a sharp or obvious bend especially in high-dimensional data (lots of features), so they should be uused as one source of evidence when deciding how many clusters are reasonable.

Let's now create a our elbow plot:

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_values, inertia_values, marker='o')

plt.title("Elbow Plot for K-means on METABRIC Data")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.show()

### <font color=0D9488>**Question 2:**</font> Look at the elbow plot. Where does the curve seem to start flattening out? Does the plot suggest one very obvious best value of `k`, or is the answer somewhat ambiguous?

**Your solution here!**

## Another way to choose k: silhouette score

The elbow plot gave us one useful clue, but it did not give a perfectly clear answer. That is common in clustering, so it helps to look at the data in more than one way.

We just used **inertia**, which tells us how tightly points are grouped around their cluster centers, so it focuses on **compactness**.

Now we will look at a different idea called **silhouette score**, which focuses more on **separation** between clusters.

The basic intuition is:
- a point should be close to other points in its own cluster
- and farther away from points in other clusters

A **higher silhouette score** usually means:
- points fit well within their own cluster
- and the clusters are more clearly separated from one another

So inertia and silhouette score tell us different things:
- **inertia** asks whether points are close to their assigned centroids
- **silhouette score** asks whether clusters are distinct from one another

Just like inertia, silhouette score is a helpful guide, not final proof.

## Step 5: Compute silhouette score for one clustering

Just like inertia, silhouette score is easier to understand if we start with one clustering first.

We will begin with `k=3`, compute the silhouette score for that clustering, and then think about what the number means.

Remember that silhouette score focuses more on how well the clusters are separated from each other.

In [ ]:
# Fit a K-means model with k = 3
kmeans_3 = KMeans(n_clusters=3, random_state=42)
labels_3 = kmeans_3.fit_predict(df)

# Compute the silhouette score for this clustering
score_3 = silhouette_score(df, labels_3)

print("Silhouette score for k = 3:")
print(f"{score_3:.3f}")

This gives us one silhouette score for one clustering.

A higher value usually means the clusters are more clearly separated, but just like inertia, one number by itself is hard to interpret. To use silhouette score well, we need to compare it across several values of `k`.


## Step 6: Compare silhouette scores across several values of `k`

Now we will compute silhouette scores for the same range of `k` values we used with inertia.

This will let us compare how cluster separation changes as we ask K-means to make more clusters.

### <font color=0D9488>**Question 3:**</font> Below shows code for our for loop. However, there are several parts that are incorrect. Identify and fix them to ensure you correctly are running K-means across multiple values of k and printing the silhouette score for each.

*Hint: there are 5 problems with 4 lines in the code below*

In [ ]:
# Your solution here!
# Fix the code below

# Values of k we want to compare
k_values = [2, 3, 4, 5, 6]

# Empty list to store the silhouette scores
silhouette_values = []

for k in silhouette_values:
    # Create the K-means model
    kmeans = KMeans(n_clusters=3, random_state=42)

    # Fit the model and get cluster labels
    cluster_labels = kmeans.inertia_(df)

    # Compute the silhouette score
    score = silhouette_score(k, k_values)

    # Add the silhouette score to your list
    silhouette_values.append(score)

print("Silhouette scores:")
for k, score in zip(k_values, silhouette_values):  # zip is a handy way to pair up information in order
    print(f"k = {k}: silhouette score = {score:.3f}")

You should have printed your 5 diffferent silhouette scores for each value of k if your code worked successfully.

Now let us plot these so we can easily visually compare them more easily across different values of `k`.

When you look at the plot, focus on which values of `k` give **higher** silhouette scores. In general, a higher silhouette score suggests that the clusters are more clearly separated from one another.

At the same time, do not assume the highest point automatically gives the one perfect answer. Just like the elbow plot, this graph is a useful guide, and it should be interpreted together with the other evidence we have.

### <font color=0D9488>**Question 4:**</font> Make a line plot of the silhouette scores, (similar to how you did the elbow plot above).

Your plot should:
- plot `k_values` on the x-axis
- plot `silhouette_values` on the y-axis
- show an 'o' marker at each point
- have a title and both axes labelled

When you are done, display the plot with `plt.show()`.

In [ ]:
# Your solution here!

### <font color=0D9488>**Question 5:**</font> Which value of `k` has the highest silhouette score? Based on this plot alone, which value of `k` would you choose, and why? Does it agree with what you saw and would choose based on inertia and the elbow plot?

**Your solution here!**

## Step 7: Set up `k = 3` and `k = 5` for visual comparison

So far, you have compared different values of `k` using inertia, elbow plots, and silhouette scores. Those tools give useful evidence, but they do not always settle the question completely.

A value like `k=2` may give the clearest separation by silhouette score, but it can also be very broad. To see whether a more detailed clustering is useful, it helps to look at a couple of larger values of `k` more closely.

Here we will focus on `k=3` and `k=5`:
- `k=3` gives a simpler grouping that is still more detailed than `k=2`
- `k=5` may capture finer structure, but it may also split the data into smaller groups

In this step, you will fit both models and save their cluster labels so we can compare them visually in PCA space.

### <font color=0D9488>**Question 6:**</font> Fit two new K-means models so we can compare `k=3` and `k=5` visually in the next step.

Complete the following tasks:
- Fit a K-means model with `k=3` and store it in a variable named `kmeans_3`
- Fit a K-means model with `k=5` and store it in a variable named `kmeans_5`
- Store the cluster labels in variables named `labels_3` and `labels_5`

*Note: Run all models with `random_state=42` for consistency with key*

In [ ]:
# Your solution here!

Before we move on to visualizing these models' clusters in PCA plots, let's print the inertia and silhouette score for `k=3` and `k=5` side by side.

This gives us one more quick way to compare the two clusterings numerically before we compare them visually.

In [ ]:
inertia_3 = kmeans_3.inertia_
inertia_5 = kmeans_5.inertia_

silhouette_3 = silhouette_score(df, labels_3)
silhouette_5 = silhouette_score(df, labels_5)

print("k = 3")
print(f"inertia: {inertia_3:.3f}")
print(f"silhouette score: {silhouette_3:.3f}\n") # \n means a literal newline, add a space line in between

print("k = 5")
print(f"inertia: {inertia_5:.3f}")
print(f"silhouette score: {silhouette_5:.3f}")

For this dataset, `k=5` has lower inertia and a higher silhouette score than `k=3`, so both of these internal measures lean toward `k=5` in this comparison.

Still, it is important to interpret those numbers carefully. We expect inertia to go down when `k` gets larger, so a lower inertia for `k=5` is not surprising by itself. The higher silhouette score is more interesting, because it suggests that `k=5` may also have somewhat clearer separation than `k=3`.

Even so, these numbers do not settle everything. A clustering can look better numerically but still be harder to describe or interpret. That is why it is useful to look at the PCA plots next and ask whether the extra detail in `k=5` seems helpful or whether it makes the grouping feel more fragmented.

## Step 8: Visualize `k = 3` and `k = 5` in PCA space

So far, the numerical results favor `k=5` over `k=3`. But in clustering, numbers are not the whole story. It is also helpful to look at the clusters visually and ask:
- Do the groups look clearly separated?
- Does the larger value of `k` reveal useful structure?
- Or does it make the clustering feel more split up and harder to describe?

To explore that, we will visualize both clusterings in PCA space, since we can't plot high dimensional data directly/clearly.

Remember:
- **PCA** helps us summarize many features using new axes
- **K-means** creates clusters based on similarity in the full feature space

So in this step, PCA is only helping us **see** the cluster assignments in two dimensions, it is not the clustering method itself.

### <font color=0D9488>**Question 7:**</font> Use PCA to create a 2D version of the METABRIC dataset for plotting, then compare the `k=3` and `k=5` clusterings.

Complete the following tasks:
- Create a PCA object with 2 components and store it in `pca`
- Use it to transform `df` and store the result in `X_pca`
- Create a dataframe named `pca_df` with columns `"PC1"` and `"PC2"`
- Add the cluster labels as columns named `"cluster_3"` and `"cluster_5"`
- Make two scatterplots:
  - one colored by `cluster_3`
  - one colored by `cluster_5`

Be sure to include titles and axis labels.

*Hint: remember `palette="deep"` is helpful in the scatterplot command if the dots show up too light to see*

In [ ]:
# Your solution here!

### <font color=0D9488>**Question 8:**</font> Compare the PCA plots for `k=3` and `k=5`. Describe any differences you notice between them. Which value of `k` do you think captures the structure of the data more usefully, and why?

**Your solution here!**

At this point, you have used three different ways to think about choosing `k`:

- The **elbow plot** of **inertia** helped you look for a point where adding more clusters stopped improving compactness by very much.
- The **silhouette score** helped you check how clearly separated the clusters were.
- The **PCA plots** gave you a visual way to compare whether different choices of `k` produced broad, simple groups or more detailed but more fragmented ones.

Together, these tools help us make a more informed decision, but they do not guarantee one single perfect answer. That is why choosing `k` is usually a matter of weighing several kinds of evidence rather than following one rule.

In practice, the best choice of `k` also depends on the scientific question we are trying to answer. If our goal is to describe a small number of broad patient groups, we might prefer a smaller value such as `k=2` or `k=3`. If we want to look for finer structure in the data, we might explore a larger value such as `k=5`.

The important idea is that clustering is not just about finding the lowest score or the highest score, clustering decisions should use several tools together, rather than relying only on the lowest or highest score.

### <font color=0D9488>**Question 9:**</font> Based on the elbow plot, silhouette scores, and PCA plots, which value of `k` would you choose for this dataset at this stage: `k=2`, `k=3`, or `k=5`? Briefly explain your choice using at least two pieces of evidence.

**Your solution here!**

## What should we conclude?

By now, you have used several different tools to think about the number of clusters:
- inertia
- elbow plots
- silhouette score
- PCA visualization

The important takeaway is that choosing `k` is often a judgment call!

Here are a few reasons:
- inertia usually improves as `k` gets larger
- elbow plots are not always sharp
- silhouette score is helpful, but not perfect
- more clusters can reveal more detail, but they can also make the results harder to interpret

In real biomedical data, clustering is usually part of an exploratory process, not a final proof of meaning step.

# Congratulations, you have completed today's notebook!

In this notebook, you learned that choosing the number of clusters is not always a one-step or one-number decision. You used internal tools such as inertia, elbow plots, and silhouette score, and you looked at the cluster patterns in PCA space. Most importantly, you practiced treating clustering as a tool for exploring structure in the data and making a reasoned judgment from multiple forms of evidence.

## Key Takeaways:
- K-means requires us to choose the number of clusters `k` ahead of time.
- Inertia measures how tightly points sit around their cluster centers.
- Elbow plots can help us look for a reasonable value of `k`, but they are not always definitive.
- Silhouette score helps measure how well-separated the clusters are.
- More than one value of `k` can be reasonable for the same dataset.
- Choosing `k` usually involves balancing numerical evidence with interpretability.

## Where This Fits in the ML Workflow
In this notebook, you followed an unsupervised learning workflow: load the data, compare possible values of `k`, evaluate the resulting clusters with internal tools, visualize the patterns, and interpret the results carefully. This kind of workflow is useful when we want to explore structure in a dataset without starting from known labels or outcomes.

Next week, you will explore a different clustering method and see how changing the method can change the groups you find.